# Blue Lodge Frontier GRPO: Multi-Track Alignment Fleet (Colab A100)

This notebook trains specialized Rank-12 LoRA adapters using **Group Relative Policy Optimization (GRPO)** with $G=12$ rollouts per prompt across 3 parallel tracks:
- **Track 1:** Tool Syntax & Schema Integrity (`curriculum_track1_syntax.jsonl`)
- **Track 2:** Git Operations & Safe File Lifecycle (`curriculum_track2_git_ops.jsonl`)
- **Track 3:** Software Phytology Living Tissue Protocol (`curriculum_track3_phytology.jsonl`)

After training, adapters are analytically fused into `Blue-Llama-27B-Champion-v5-Iteration5-Fused-LoRA.gguf`.

## 1. Environment Setup & Dependencies

In [ ]:
# Verify GPU
!nvidia-smi

# Install PyTorch, GGUF, requests, numpy
!pip install -q --upgrade pip
!pip install -q torch numpy gguf requests

## 2. Select Track & Launch GRPO Training ($G=12$, -np 12)
Set `TRACK_ID` to `1`, `2`, or `3` depending on which Colab instance this is.

In [ ]:
import os

TRACK_ID = 1  # @param [1, 2, 3] {type:"integer"}
ENDPOINT = "http://127.0.0.1:8080"  # @param {type:"string"}
STEPS = 40  # @param {type:"integer"}
GROUP_SIZE = 12  # @param {type:"integer"}

track_configs = {
    1: {
        "curriculum": "data/training/curriculum_track1_syntax.jsonl",
        "output": "models/Blue-Llama-27B-Champion-v5-LoRA-Syntax.gguf",
        "ckpt": "models/checkpoints_syntax"
    },
    2: {
        "curriculum": "data/training/curriculum_track2_git_ops.jsonl",
        "output": "models/Blue-Llama-27B-Champion-v5-LoRA-GitOps.gguf",
        "ckpt": "models/checkpoints_gitops"
    },
    3: {
        "curriculum": "data/training/curriculum_track3_phytology.jsonl",
        "output": "models/Blue-Llama-27B-Champion-v5-LoRA-Phytology.gguf",
        "ckpt": "models/checkpoints_phytology"
    }
}

cfg = track_configs[TRACK_ID]
os.makedirs(cfg["ckpt"], exist_ok=True)

cmd = f"""python3 scripts/colab_training/train_blue_lodge_grpo.py \
  --curriculum {cfg['curriculum']} \
  --steps {STEPS} \
  --group_size {GROUP_SIZE} \
  --parallel {GROUP_SIZE} \
  --rank 12 \
  --alpha 16.0 \
  --endpoint {ENDPOINT} \
  --ckpt_dir {cfg['ckpt']} \
  --output {cfg['output']}"""

print(f"Executing Track {TRACK_ID} training...")
!{cmd}

## 3. Analytical LoRA Fusion Engine (Iteration 5)
Run this once all 3 track GGUFs are available to fuse them together with the STEM, ARC, and Exploit adapters.

In [ ]:
!python3 scripts/colab_training/fuse_iteration5.py --models_dir models

## 4. Run BlueLodgeBench Verification Gauntlet

In [ ]:
!python3 scripts/quant_research/05_evaluate_blue_lodge_benchmark.py \
  --endpoint http://127.0.0.1:8080 \
  --model_label "Blue-Llama-27B-Champion-v5-Iteration5"